In [402]:

import pandas as pd
import numpy as np
import requests as r
import json as j
import ast as a
import pprint as pp




In [69]:

odds_api_key = ''

sport = 'baseball_mlb' # use the sport_key from the /sports endpoint below, or use 'upcoming' to see the next 8 games across all sports

regions = 'us' #multiple can be specified if comma delimited

markets = 'h2h,spreads'

bookmakers = 'us,us2'

odds_format = 'american'

date_format = 'iso'

all_format = 'true'
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
#
# First get a list of in-season sports
#   The sport 'key' from the response can be used to get odds in the next request
#
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

sports_response = r.get(
    'https://api.the-odds-api.com/v4/sports', 
    params={
        'api_key': odds_api_key,
        'regions': regions,
        'sport': sport,
        'all': all_format
    }
)


if sports_response.status_code != 200:
    print(f'Failed to get sports: status_code {sports_response.status_code}, response body {sports_response.text}')

else:
    print('List of in season sports:', sports_response.json())
    #dump the list of dictionaries provided by json output via api call in to a python data structure for further parsing
    sports_data = j.dumps(sports_response.json())
    #turn string to list
    sports_load_data = j.loads(sports_data)
    #iterate through list to get dictionaries
    df_sports = pd.DataFrame(sports_load_data)




    # Check the usage quota
    print('Remaining requests', sports_response.headers['x-requests-remaining'])
    print('Used requests', sports_response.headers['x-requests-used'])

df_sports.to_csv('list_of_sports.csv',index=True)


List of in season sports: [{'key': 'americanfootball_cfl', 'group': 'American Football', 'title': 'CFL', 'description': 'Canadian Football League', 'active': False, 'has_outrights': False}, {'key': 'americanfootball_ncaaf', 'group': 'American Football', 'title': 'NCAAF', 'description': 'US College Football', 'active': True, 'has_outrights': False}, {'key': 'americanfootball_ncaaf_championship_winner', 'group': 'American Football', 'title': 'NCAAF Championship Winner', 'description': 'US College Football Championship Winner', 'active': True, 'has_outrights': True}, {'key': 'americanfootball_nfl', 'group': 'American Football', 'title': 'NFL', 'description': 'US Football', 'active': False, 'has_outrights': False}, {'key': 'americanfootball_nfl_preseason', 'group': 'American Football', 'title': 'NFL Preseason', 'description': 'US Football', 'active': False, 'has_outrights': False}, {'key': 'americanfootball_nfl_super_bowl_winner', 'group': 'American Football', 'title': 'NFL Super Bowl Winn

In [70]:
odds_response = r.get(
    f'https://api.the-odds-api.com/v4/sports/{sport}/odds',
    params={
        'api_key': odds_api_key,
        'regions': regions,
        'markets': markets,
        'oddsFormat': odds_format,
        'dateFormat': date_format,
    }
)

if odds_response.status_code != 200:
    print(f'Failed to get sports: status_code {odds_response.status_code}, response body {odds_response.text}')

else:
    #dump the list of dictionaries provided by json output via api call in to a python data structure for further parsing
    odds_data = odds_response.json()


    
    

    # Check the usage quota
    print('Remaining requests', odds_response.headers['x-requests-remaining'])
    print('Used requests', odds_response.headers['x-requests-used'])

Remaining requests 498
Used requests 2


In [5]:
rows_list = []
for i in odds_data:

    event_id = i['id']
    sport = i['sport_title']
    match_time = i['commence_time']
    home_team = i['home_team']
    away_team = i['away_team']

    for bookmakers in i['bookmakers']:

        bookmaker = bookmakers['title']
        bookmaker_last_update = bookmakers['last_update']

        for markets in bookmakers['markets']:
            
            market_event = markets['key']
            market_event_last_update = markets['last_update']

            for outcomes in markets['outcomes']:

                dictionary = {

                       'event_id': event_id,
                       'sport':sport,
                       'match_time':match_time,
                       'home_team':home_team,
                       'away_team':away_team,
                       'bookmaker':bookmaker,
                       'bookmaker_updated':bookmaker_last_update,
                       'market_event':market_event,
                       'market_event_update':market_event_last_update,
                       'outcome_name':outcomes['name'],
                       'price':outcomes['price'],
                       'points':outcomes.get('point')
                       

                      }
                
                rows_list.append(dictionary)

                
#df_full = pd.concat([pd.DataFrame([row]) for row in rows_list], ignore_index=True)
pd.DataFrame(rows_list).head(30)
        

#df_full.head()
    

bookmaker_actual = pd.DataFrame(rows_list)

bookmaker_actual.to_csv('bookmakers_actual.csv',index=True)


In [19]:
pivotted_odds = pd.pivot_table(bookmaker_actual,values = ['price','points'],index= ['event_id','market_event','outcome_name'],aggfunc=['mean','min','max']).reset_index()
pivotted_odds.columns = ['_'.join(col) for col in pivotted_odds.columns]
pivotted_odds.columns = [col[0:len(col)-1] if col[-1] == '_' else col for col in pivotted_odds.columns]
pivotted_odds.to_csv('pivotted_odds.csv',index=True)

pivotted_odds_and_bookmakers = pd.merge(bookmaker_actual,pivotted_odds,how='left',on=['event_id','market_event','outcome_name'])
pivotted_odds_and_bookmakers

,event_id,sport,match_time,home_team,away_team,bookmaker,bookmaker_updated,market_event,market_event_update,outcome_name,price,points,mean_points,mean_price,min_points,min_price,max_points,max_price
0,9aac4fc2c2231d9a8eab2a40c2a8cddf,MLB,2024-03-30T00:11:09Z,Houston Astros,New York Yankees,DraftKings,2024-03-30T03:03:35Z,h2h,2024-03-30T03:03:35Z,Houston Astros,3500,NaN,NaN,2940.000000,NaN,1900,NaN,4300
1,9aac4fc2c2231d9a8eab2a40c2a8cddf,MLB,2024-03-30T00:11:09Z,Houston Astros,New York Yankees,DraftKings,2024-03-30T03:03:35Z,h2h,2024-03-30T03:03:35Z,New York Yankees,-50000,NaN,NaN,-35600.000000,NaN,-100000,NaN,-8000
2,9aac4fc2c2231d9a8eab2a40c2a8cddf,MLB,2024-03-30T00:11:09Z,Houston Astros,New York Yankees,DraftKings,2024-03-30T03:03:35Z,spreads,2024-03-30T03:03:35Z,Houston Astros,154,4.5,4.944444,7.777778,4.5,-260,5.5,200
3,9aac4fc2c2231d9a8eab2a40c2a8cddf,MLB,2024-03-30T00:11:09Z,Houston Astros,New York Yankees,DraftKings,2024-03-30T03:03:35Z,spreads,2024-03-30T03:03:35Z,New York Yankees,-200,-4.5,-4.944444,-101.777778,-5.5,-260,-4.5,190
4,9aac4fc2c2231d9a8eab2a40c2a8cddf,MLB,2024-03-30T00:11:09Z,Houston Astros,New York Yankees,MyBookie.ag,2024-03-30T03:03:35Z,spreads,2024-03-30T03:03:35Z,Houston Astros,-225,5.5,4.944444,7.777778,4.5,-260,5.5,200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
977,48f6a5704c93970245f3f0c01292cda2,MLB,2024-03-31T01:40:00Z,Seattle Mariners,Boston Red Sox,PointsBet (US),2024-03-30T03:03:35Z,spreads,2024-03-30T03:03:35Z,Seattle Mariners,140,-1.5,-1.458333,122.583333,-1.5,-104,-1.0,148
978,48f6a5704c93970245f3f0c01292cda2,MLB,2024-03-31T01:40:00Z,Seattle Mariners,Boston Red Sox,BetUS,2024-03-30T03:03:35Z,h2h,2024-03-30T03:03:35Z,Boston Red Sox,130,NaN,NaN,129.642857,NaN,120,NaN,133
979,48f6a5704c93970245f3f0c01292cda2,MLB,2024-03-31T01:40:00Z,Seattle Mariners,Boston Red Sox,BetUS,2024-03-30T03:03:35Z,h2h,2024-03-30T03:03:35Z,Seattle Mariners,-145,NaN,NaN,-149.642857,NaN,-157,NaN,-144
980,48f6a5704c93970245f3f0c01292cda2,MLB,2024-03-31T01:40:00Z,Seattle Mariners,Boston Red Sox,BetUS,2024-03-30T03:03:35Z,spreads,2024-03-30T03:03:35Z,Boston Red Sox,-165,1.5,1.458333,-165.833333,1.0,-180,1.5,-118


In [857]:
mlb_api_key = 'FWGQoAvCrTayY6TUyndhH1VD5bcNfrXu8ht7U4e1'
format = 'json'
language_code = 'en'
access_level = 'trial'
season_type = 'REG'



baseball_teams_response = r.get(
    f'https://api.sportradar.com/mlb/v7/en/league/teams.json?',
    params={
        'api_key': mlb_api_key,
        'access_level': access_level
      
    }
)

if baseball_teams_response.status_code != 200:
    print(f'Failed to get sports: status_code {baseball_teams_response.status_code}, response body {baseball_teams_response.text}')

else:
    #dump the list of dictionaries provided by json output via api call in to a python data structure for further parsing
    baseball_teams_data = baseball_teams_response.json()

league_name = baseball_teams_data['league']['alias']

In [858]:
mlb_teams = []
for i in baseball_teams_data['teams']:
    team_name = i.get('name')
    market_name = i.get('market','No Market')
    team_id = i.get('id')

    dictionary = {

        'league_name':league_name,
        'team_name':team_name,
        'market_name':market_name,
        'team_id':team_id
    }

    mlb_teams.append(dictionary)

baseball_teams = pd.DataFrame(mlb_teams)

baseball_teams = baseball_teams[baseball_teams.market_name != 'No Market']



In [907]:

team_id = '44671792-dc02-4fdd-a5ad-f5f17edaa9d7'
season_year = '2023'

baseball_seasonal_stats = r.get(
    f'https://api.sportradar.com/mlb/{access_level}/v7/{language_code}/seasons/{season_year}/{season_type}/teams/{team_id}/statistics.{format}',

    params = {

        'api_key':mlb_api_key,
        'language_code':language_code,
        'access_level':access_level,
        'year':season_year,
        'team_id':team_id,
        'season_type':season_type,
        'format':format
        }
    
    
)

if baseball_seasonal_stats.status_code != 200:
    print(f'Failed to get sports: status_code {baseball_seasonal_stats.status_code}, response body {baseball_seasonal_stats.text}')

else:
    #dump the list of dictionaries provided by json output via api call in to a python data structure for further parsing
    baseball_seasonal_stats = baseball_seasonal_stats.json()



In [908]:
team_name = baseball_seasonal_stats.get('name','No Team Name')
team_market = baseball_seasonal_stats.get('market','No Market')
team_id = baseball_seasonal_stats.get('id','No Team ID')
team_season_id = baseball_seasonal_stats['season'].get('id','No Season ID')
team_year = baseball_seasonal_stats['season'].get('year','0001')
team_stats_category = []
for label, stats in baseball_seasonal_stats['statistics'].items():

    stat_category = label

    for key, value in stats.items():

        stat_sub_category = key

        for i in value:
            
            statistic_parent_name = i
            
            if type(value[i]) != dict:
                dictionary = {
                              'team_id':team_id,
                              'season_id':team_season_id,
                              'stat_category':label,
                              'stat_sub_category':key,
                              'statistical_parent_name':statistic_parent_name,
                               'value':value[i]
                                  }
                team_stats_category.append(dictionary)
            
        
            elif type(value[i]) == dict:
                
                for j in value[i]:
                    
                    dictionary = {
                                  'team_id':team_id,
                                  'season_id':team_season_id,
                                  'stat_category':label,
                                  'stat_sub_category':key,
                                  'statistical_parent_name':statistic_parent_name,
                                   j:value[i].get(j,0)
                                  }
                    team_stats_category.append(dictionary) 
            


team_df = pd.DataFrame(team_stats_category)
#pivotted_team_df = pd.pivot_table(team_df,index = ['team_id','season_id','stat_category','stat_sub_category','statistical_parent_name'],aggfunc='sum').reset_index()


team_df_fielding = team_df[team_df['stat_category'] == 'fielding'].reset_index(drop=True)
team_df_hitting = team_df[team_df['stat_category'] == 'hitting'].reset_index(drop=True)
team_df_pitching = team_df[team_df['stat_category'] == 'pitching'].reset_index(drop=True)


team_df_fielding.dropna(axis = 1, how = 'all',inplace=True)

team_df_hitting.dropna(axis = 1, how = 'all',inplace=True)

team_df_pitching.dropna(axis = 1, how = 'all',inplace=True)

#team_df.to_csv('team_stats.csv', index=True)
#team_df_fielding.to_csv('fielding.csv',index=True)
#team_df_hitting.to_csv('hitting.csv',index=True)
#team_df_pitching.to_csv('pitching.csv',index=True)



In [909]:

def get_max_array_values_for_stats(df, ls, ls1):

   value_location = df.columns.get_loc('value')
   columns_to_float = df.columns[value_location:]
   df[columns_to_float] = df[columns_to_float].astype(float)
   
   for i in df.values:

      
      try:
         ls.append(np.nanmax(i[value_location:]))
         
      except TypeError:
         print(TypeError)
      

   for df_row_index, rows in df.iterrows():
      

      counter = 0

      for name, value in rows.items():
            
            if counter >= value_location and not pd.isna(value):
                #print(name,value)
                ls1.append(rows['statistical_parent_name']+'_'+name)
            else:
                pass
               
            counter += 1

   df.loc[:,'value'] = ls
   df.loc[:,'statistical_parent_name'] = ls1
   
   df = df.iloc[:,:value_location+1]
   

   
   return df
   



In [910]:
get_max_array_values_for_stats(df = team_df_fielding, ls = [],ls1 = [])
get_max_array_values_for_stats(df = team_df_hitting, ls = [],ls1 = [])
get_max_array_values_for_stats(df = team_df_pitching, ls = [],ls1 = [])

,team_id,season_id,stat_category,stat_sub_category,statistical_parent_name,value
0,44671792-dc02-4fdd-a5ad-f5f17edaa9d7,c8bca8f4-fb0f-44ed-95e0-8ced816ae8d8,pitching,overall,oba_value,0.2760
1,44671792-dc02-4fdd-a5ad-f5f17edaa9d7,c8bca8f4-fb0f-44ed-95e0-8ced816ae8d8,pitching,overall,lob_value,2462.0000
2,44671792-dc02-4fdd-a5ad-f5f17edaa9d7,c8bca8f4-fb0f-44ed-95e0-8ced816ae8d8,pitching,overall,era_value,4.7890
3,44671792-dc02-4fdd-a5ad-f5f17edaa9d7,c8bca8f4-fb0f-44ed-95e0-8ced816ae8d8,pitching,overall,k9_value,7.6590
4,44671792-dc02-4fdd-a5ad-f5f17edaa9d7,c8bca8f4-fb0f-44ed-95e0-8ced816ae8d8,pitching,overall,whip_value,1.4555
...,...,...,...,...,...,...
239,44671792-dc02-4fdd-a5ad-f5f17edaa9d7,c8bca8f4-fb0f-44ed-95e0-8ced816ae8d8,pitching,bullpen,games_win,28.0000
240,44671792-dc02-4fdd-a5ad-f5f17edaa9d7,c8bca8f4-fb0f-44ed-95e0-8ced816ae8d8,pitching,bullpen,games_loss,27.0000
241,44671792-dc02-4fdd-a5ad-f5f17edaa9d7,c8bca8f4-fb0f-44ed-95e0-8ced816ae8d8,pitching,bullpen,games_save,36.0000
242,44671792-dc02-4fdd-a5ad-f5f17edaa9d7,c8bca8f4-fb0f-44ed-95e0-8ced816ae8d8,pitching,bullpen,games_hold,89.0000


In [911]:
team_df_fielding_pivotted = pd.pivot_table(team_df_fielding,index=['team_id','season_id','stat_category','stat_sub_category'],columns=['statistical_parent_name'],values = 'value',aggfunc='sum')
team_df_hitting_pivotted = pd.pivot_table(team_df_hitting,index = ['team_id','season_id','stat_category','stat_sub_category'], columns = ['statistical_parent_name'], values = 'value',aggfunc = 'sum')
team_df_pitching_pivotted = pd.pivot_table(team_df_pitching,index = ['team_id','season_id','stat_category','stat_sub_category'],columns = ['statistical_parent_name'],values = 'value',aggfunc = 'sum')                                                                                                          


In [947]:
players = []
player_stats = []
player_sub_category_stats_list = []

for i in baseball_seasonal_stats['players']:
    
    preferred_name = i['preferred_name']
    first_name = i['first_name']
    last_name = i['last_name']
    player_id = i['id']
    position = i['position']
    primary_position = i['primary_position']

    dictionary = {
                  'id':player_id,
                  'player_first_name':first_name,
                  'player_last_name':last_name,
                  'position':position,
                  'primary_position':primary_position

                 }
    
    players.append(dictionary)

    for label, stats in i['statistics'].items():
        
        
        stat_category = label

        for key, value in stats.items():
            stat_sub_category = key
            if type(value) == list:
                pass
            else:
                for sub_key, sub_value in value.items():
                    if type(sub_value) == dict:
                       player_specific_stat_category = sub_key
                       for k, v in sub_value.items():
                          
                           dictionary_1 = {
                                         'player_id':player_id,
                                         'stat_category':stat_category,
                                         'stat_sub_category':stat_sub_category,
                                         'player_specific_stat_category':player_specific_stat_category,
                                         'player_specific_stat_category_name':k,
                                         'player_specific_stat_category_name_value':v

                               
                                        }
                           player_sub_category_stats_list.append(dictionary_1)
                    else:
                            dictionary_2 = {
                                            'player_id':player_id,
                                            'stat_category':stat_category,
                                            'stat_sub_category':stat_sub_category,
                                            'player_specific_stat_category':player_specific_stat_category,
                                            'player_specific_stat_category':sub_key,
                                            'player_specific_stat_category_value':sub_value

                                           }
                            
                            player_stats.append(dictionary_2)
                        




[{'id': 'ecee463a-f7fb-43ad-9a71-d20c3e4c5d38',
  'player_first_name': 'Joseph',
  'player_last_name': 'Romero',
  'position': 'P',
  'primary_position': 'RP'},
 {'id': '6162380d-5201-4e3c-a1a3-0d3db5afc356',
  'player_first_name': 'Masyn',
  'player_last_name': 'Winn',
  'position': 'IF',
  'primary_position': 'SS'},
 {'id': '8babdbd4-d607-4bb0-a6c1-708c247c08ed',
  'player_first_name': 'Brendan',
  'player_last_name': 'Donovan',
  'position': 'OF',
  'primary_position': 'LF'},
 {'id': '65fc46ca-350d-4eaf-bc40-8851c3378bc6',
  'player_first_name': 'Tyler',
  'player_last_name': "O'Neill",
  'position': 'OF',
  'primary_position': 'RF'},
 {'id': 'cf302aef-cb3a-4626-ba44-a9bc7bbb3f5b',
  'player_first_name': 'Paul',
  'player_last_name': 'DeJong',
  'position': 'IF',
  'primary_position': 'SS'},
 {'id': 'f3719f44-d0f6-4d30-9bb3-22a3f315aae7',
  'player_first_name': 'Génesis',
  'player_last_name': 'Cabrera',
  'position': 'P',
  'primary_position': 'RP'},
 {'id': '96f75024-d2cb-4c0e-a88